# MRR rescoring to match Neel Nanda's methodology

Re-scores both fitted lenses using **Mean Reciprocal Rank (MRR)** across all layers, as Neel Nanda and Anthropic do — not the pass@k that our earlier code used.

MRR is defined per-item as: `1 / (best rank of correct token across layers)`. Then averaged over items. Higher = better.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'git+https://github.com/anthropics/jacobian-lens.git',
    '--upgrade', 'transformers>=5.5'])
import torch
print(torch.cuda.get_device_name(0), '| torch:', torch.__version__)

In [ ]:
# Both fitted lenses are attached as kernel data sources.
import glob, os
for root,dirs,files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.pt'):
            print(os.path.join(root,f))

In [ ]:
import torch, transformers, jlens, json, pathlib

def score_mrr(MODEL, LENS_PATH, OUT_NAME):
    print(f'\n===== {MODEL} =====')
    hf_model = transformers.AutoModelForCausalLM.from_pretrained(
        MODEL, torch_dtype=torch.bfloat16, attn_implementation='sdpa', low_cpu_mem_usage=True
    ).to('cuda')
    tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL)
    model = jlens.from_hf(hf_model, tokenizer, compile=True)
    lens = jlens.JacobianLens.load(LENS_PATH)
    print(f'  n_layers={model.n_layers} d_model={model.d_model} lens.n_prompts={lens.n_prompts}')

    # Load evals
    if not pathlib.Path('/tmp/jl').exists():
        subprocess.run(['git','clone','-q','--depth=1','https://github.com/anthropics/jacobian-lens.git','/tmp/jl'])
    EVALS_DIR = pathlib.Path('/tmp/jl/data/evaluations')

    def tokens_of(word):
        ids=[]
        for v in {word,' '+word,word.lower(),' '+word.lower()}:
            e = tokenizer(v, add_special_tokens=False).input_ids
            if len(e)==1: ids.append(e[0])
        return sorted(set(ids))

    def best_rank(lens_logits, token_ids):
        """Best (min) rank of the given token IDs across all layers. 1-indexed."""
        if not token_ids: return 10**9
        best = 10**9
        for _, logits in lens_logits.items():
            order = logits[0].argsort(descending=True)
            rank_lookup = torch.empty_like(order)
            rank_lookup[order] = torch.arange(order.numel(), device=order.device)
            cand = rank_lookup[torch.tensor(token_ids, device=order.device)]
            r = int(cand.min().item())+1
            if r<best: best=r
        return best

    all_results = {}
    for path in sorted(EVALS_DIR.glob('lens-eval-*.json')):
        data = json.load(path.open())
        items = data['items']
        per_item_rrs = []       # reciprocal ranks per item (avg over intermediates)
        per_item_top1 = []      # top-1 hit fraction per item
        per_item_top10 = []     # top-10 hit fraction per item (matches old scoring)
        n_dropped = 0
        for item in items:
            try:
                lens_logits, _, _ = lens.apply(model, item['prompt'], positions=[-1])
            except Exception:
                continue
            inters = item['intermediates']
            if not isinstance(inters, list): inters = [inters]
            rrs, t1s, t10s = [], [], []
            for inter in inters:
                key = inter if isinstance(inter, str) else (
                    inter.get('synonyms', [inter.get('key','')])[0] if isinstance(inter, dict) else inter[0]
                )
                tok_ids = tokens_of(str(key))
                if not tok_ids:
                    n_dropped += 1
                    continue
                r = best_rank(lens_logits, tok_ids)
                rrs.append(1.0/r)
                t1s.append(1.0 if r<=1 else 0.0)
                t10s.append(1.0 if r<=10 else 0.0)
            if rrs:
                per_item_rrs.append(sum(rrs)/len(rrs))
                per_item_top1.append(sum(t1s)/len(t1s))
                per_item_top10.append(sum(t10s)/len(t10s))
        n = len(per_item_rrs)
        result = {
            'n_items_scored': n,
            'n_multitoken_dropped': n_dropped,
            'MRR': (sum(per_item_rrs)/max(1,n)),
            'top1': (sum(per_item_top1)/max(1,n)),
            'top10': (sum(per_item_top10)/max(1,n)),
        }
        all_results[path.stem] = result
        print(f"  {path.stem:<28} n={n:>3}  MRR={result['MRR']:.4f}  top-1={result['top1']:.4f}  top-10={result['top10']:.4f}")

    # Free VRAM before next model
    del model, lens, hf_model, tokenizer
    torch.cuda.empty_cache()
    return all_results

# Find both lens files under /kaggle/input
cands = glob.glob('/kaggle/input/**/*.pt', recursive=True)
qwen_lens = next(c for c in cands if 'qwen' in c.lower())
gemma_lens = next(c for c in cands if 'gemma' in c.lower() or 'e2b' in c.lower())
print('Qwen lens: ', qwen_lens)
print('Gemma lens:', gemma_lens)

results = {}
results['Qwen/Qwen2.5-3B-Instruct']  = score_mrr('Qwen/Qwen2.5-3B-Instruct', qwen_lens, 'qwen')
results['google/gemma-4-E2B-it']      = score_mrr('google/gemma-4-E2B-it',    gemma_lens, 'gemma')

pathlib.Path('/kaggle/working/mrr_rescore.json').write_text(json.dumps(results, indent=2))
print('\nSaved.')
for m,r in results.items():
    print(f'\n{m}:')
    for eval_name, sc in r.items():
        print(f"  {eval_name:<30} MRR={sc['MRR']:.4f} top1={sc['top1']:.4f} top10={sc['top10']:.4f}")